# 03. SARIMA 모델

## 목표
- ACF/PACF 분석으로 SARIMA 차수 탐색
- AIC 기준 후보 차수 비교로 상품군별 최적 차수 결정
- SARIMA (순수) vs SARIMAX (외생변수) 비교
- 잔차 분석으로 모델 적합성 검증
- Validation 기간 예측 + 신뢰구간 + 베이스라인 대비 개선율

In [1]:
import sys

sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats
import warnings

warnings.filterwarnings("ignore")

fm._load_fontmanager(try_read_cache=False)
sns.set_style("whitegrid")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

from src.models.sarima_model import SARIMAModel
from src.evaluation import evaluate_model, compare_models

print("Setup 완료")

Setup 완료


In [2]:
full = pd.read_csv("../data/processed/full_data.csv", parse_dates=["date"])
train = pd.read_csv("../data/processed/train.csv", parse_dates=["date"])
val = pd.read_csv("../data/processed/val.csv", parse_dates=["date"])
baseline = pd.read_csv("../outputs/results/baseline_results.csv")

families = full["family"].unique().tolist()
EXOG_COLS = ["oil_price", "is_holiday", "onpromotion"]

print(f"Train: {train.shape}, Val: {val.shape}")
print(f"상품군: {families}")
print(f"외생변수: {EXOG_COLS}")

Train: (7285, 19), Val: (905, 19)
상품군: ['BEVERAGES', 'CLEANING', 'DAIRY', 'GROCERY I', 'PRODUCE']
외생변수: ['oil_price', 'is_holiday', 'onpromotion']


---
## 1. ACF / PACF 분석

1차 차분한 시계열의 ACF/PACF를 통해 ARIMA 차수의 힌트를 얻는다.
- **ACF 절단(cutoff)**: MA 차수(q) 힌트
- **PACF 절단**: AR 차수(p) 힌트
- **lag=7 배수**에서 스파이크: 주간 계절성 확인

In [3]:
import os

FIG_DIR = os.path.abspath(
    os.path.join(os.path.dirname("__file__"), "..", "outputs", "figures")
)
RES_DIR = os.path.abspath(
    os.path.join(os.path.dirname("__file__"), "..", "outputs", "results")
)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

fig, axes = plt.subplots(len(families), 2, figsize=(16, 3 * len(families)))

for i, family in enumerate(families):
    ts = train[train["family"] == family].set_index("date")["sales"].asfreq("D")
    ts = ts.interpolate(method="linear")
    ts_diff = ts.diff().dropna()

    plot_acf(ts_diff, ax=axes[i, 0], lags=40, alpha=0.05)
    axes[i, 0].set_title(f"{family} - ACF (1차 차분)", fontsize=10)

    plot_pacf(ts_diff, ax=axes[i, 1], lags=40, alpha=0.05, method="ywm")
    axes[i, 1].set_title(f"{family} - PACF (1차 차분)", fontsize=10)

plt.suptitle("ACF / PACF (1차 차분 시계열)", fontsize=13, fontweight="bold")
plt.tight_layout()
save_path = os.path.join(FIG_DIR, "12_acf_pacf.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"저장: {save_path}")

저장: E:\프로젝트\부족한 프로젝트 시작\수요예측 시스템\outputs\figures\12_acf_pacf.png


### ACF/PACF 해석

- lag=7, 14, 21... 에서 뚜렷한 스파이크 → 주간 계절성(m=7) 확인
- ACF가 서서히 감소 → AR 성분 존재
- PACF가 lag=1~2에서 유의 → p=1~2 시도

---
## 2. SARIMA 차수 결정

ACF/PACF 분석과 Day 2 정상성 검정 결과를 기반으로 차수를 결정한다.

**차수 결정 근거:**
- `d=1`: ADF 검정에서 대부분 비정상 → 1차 차분 필요
- `D=1`: 주간 패턴이 강하므로 계절 차분 1회
- `m=7`: 주간 계절성
- `p, q`: PACF/ACF cutoff에서 1~2 시도
- `P, Q`: 계절 성분은 보수적으로 1

**탐색 전략:** 각 상품군에 AIC 기준으로 후보 차수를 비교하여 최적 선택

In [4]:
import time
from statsmodels.tsa.statespace.sarimax import SARIMAX

# AIC 기반 차수 탐색: 후보 차수들을 비교
CANDIDATES = [
    ((1, 1, 1), (1, 1, 1, 7)),
    ((2, 1, 1), (1, 1, 1, 7)),
    ((1, 1, 2), (1, 1, 1, 7)),
    ((2, 1, 2), (1, 1, 1, 7)),
    ((1, 1, 1), (1, 1, 0, 7)),
    ((1, 1, 1), (0, 1, 1, 7)),
]

auto_results = {}
print("AIC 기반 차수 탐색 (6개 후보)...")
print("=" * 70)

for family in families:
    ts = train[train["family"] == family].set_index("date")["sales"].asfreq("D")
    ts = ts.interpolate(method="linear")

    best_aic = np.inf
    best_order = None
    best_seasonal = None
    start = time.perf_counter()

    for order, seasonal in CANDIDATES:
        try:
            model = SARIMAX(
                ts,
                order=order,
                seasonal_order=seasonal,
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            result = model.fit(disp=False, maxiter=200)
            if result.aic < best_aic:
                best_aic = result.aic
                best_order = order
                best_seasonal = seasonal
        except Exception:
            continue

    elapsed = time.perf_counter() - start
    auto_results[family] = {
        "order": best_order,
        "seasonal_order": best_seasonal,
        "aic": best_aic,
        "time_sec": elapsed,
    }

    print(
        f"{family:15s} → order={best_order}, seasonal={best_seasonal}, "
        f"AIC={best_aic:.1f}, 시간={elapsed:.1f}초"
    )

print("=" * 70)
print("차수 탐색 완료")

AIC 기반 차수 탐색 (6개 후보)...


BEVERAGES       → order=(2, 1, 2), seasonal=(1, 1, 1, 7), AIC=32762.1, 시간=3.3초


CLEANING        → order=(2, 1, 1), seasonal=(1, 1, 1, 7), AIC=29833.9, 시간=3.8초


DAIRY           → order=(1, 1, 1), seasonal=(0, 1, 1, 7), AIC=28400.6, 시간=2.6초


GROCERY I       → order=(2, 1, 2), seasonal=(1, 1, 1, 7), AIC=33945.4, 시간=2.9초


PRODUCE         → order=(2, 1, 2), seasonal=(1, 1, 1, 7), AIC=32107.3, 시간=2.2초
차수 탐색 완료


---
## 3. SARIMA vs SARIMAX 모델 학습

각 상품군에 대해 AIC 기반 탐색으로 결정한 차수로 2가지 모델을 비교:
1. **SARIMA**: 순수 시계열 (외생변수 없음)
2. **SARIMAX**: 외생변수 포함 (oil_price, is_holiday, onpromotion)

In [5]:
sarima_models = {}  # family -> SARIMAModel (순수)
sarimax_models = {}  # family -> SARIMAModel (외생변수)
all_results = []

for family in families:
    order = auto_results[family]["order"]
    seasonal = auto_results[family]["seasonal_order"]
    print(f"\n{family} [{order} x {seasonal}]")

    # --- SARIMA (순수) ---
    m1 = SARIMAModel(order=order, seasonal_order=seasonal, exog_cols=None)
    m1.fit(train, family)

    val_sub = val[val["family"] == family].sort_values("date")
    steps = len(val_sub)
    pred1 = m1.predict(steps=steps)

    y_true = val_sub["sales"].values[: len(pred1)]
    y_pred1 = pred1["forecast"].values[: len(y_true)]

    r1 = evaluate_model(
        y_true,
        y_pred1,
        "SARIMA",
        family,
        train_time=m1.train_time_,
        predict_time=m1.predict_time_,
    )
    r1["aic"] = m1.get_aic()
    all_results.append(r1)
    sarima_models[family] = m1

    print(
        f"  SARIMA:  MAPE={r1['mape']:.2f}%, RMSE={r1['rmse']:.0f}, AIC={r1['aic']:.0f}, "
        f"학습={m1.train_time_:.1f}초"
    )

    # --- SARIMAX (외생변수) ---
    m2 = SARIMAModel(order=order, seasonal_order=seasonal, exog_cols=EXOG_COLS)
    m2.fit(train, family)

    pred2 = m2.predict(steps=steps, val_df=val)
    y_pred2 = pred2["forecast"].values[: len(y_true)]

    r2 = evaluate_model(
        y_true,
        y_pred2,
        "SARIMAX",
        family,
        train_time=m2.train_time_,
        predict_time=m2.predict_time_,
    )
    r2["aic"] = m2.get_aic()
    all_results.append(r2)
    sarimax_models[family] = m2

    print(
        f"  SARIMAX: MAPE={r2['mape']:.2f}%, RMSE={r2['rmse']:.0f}, AIC={r2['aic']:.0f}, "
        f"학습={m2.train_time_:.1f}초"
    )

sarima_df = compare_models(all_results)
print("\n모델 학습 완료")


BEVERAGES [(2, 1, 2) x (1, 1, 1, 7)]


  SARIMA:  MAPE=74.03%, RMSE=73701, AIC=32762, 학습=1.3초


  SARIMAX: MAPE=49.70%, RMSE=58736, AIC=32730, 학습=0.9초

CLEANING [(2, 1, 1) x (1, 1, 1, 7)]


  SARIMA:  MAPE=153.74%, RMSE=20400, AIC=29834, 학습=1.2초


  SARIMAX: MAPE=139.78%, RMSE=20910, AIC=30166, 학습=1.4초

DAIRY [(1, 1, 1) x (0, 1, 1, 7)]


...

  SARIMAX: MAPE=89.45%, RMSE=15253, AIC=28656, 학습=0.6초

GROCERY I [(2, 1, 2) x (1, 1, 1, 7)]


  SARIMA:  MAPE=145.81%, RMSE=124338, AIC=33945, 학습=0.9초


  SARIMAX: MAPE=65.24%, RMSE=50685, AIC=33911, 학습=1.5초

PRODUCE [(2, 1, 2) x (1, 1, 1, 7)]


  SARIMA:  MAPE=84.69%, RMSE=33466, AIC=32107, 학습=1.1초


  SARIMAX: MAPE=79.78%, RMSE=29559, AIC=31961, 학습=2.9초

모델 학습 완료


### 3.1 SARIMA vs SARIMAX 비교표

In [6]:
print("SARIMA vs SARIMAX 성능 비교:")
print(
    sarima_df[
        ["model", "family", "mape", "rmse", "mae", "aic", "train_time_sec"]
    ].to_string(index=False, float_format="{:.2f}".format)
)

# 상품군별 더 나은 모델 선택
print("\n상품군별 최적 모델 (MAPE 기준):")
best_per_family = sarima_df.loc[sarima_df.groupby("family")["mape"].idxmin()]
for _, row in best_per_family.iterrows():
    print(f"  {row['family']:15s} → {row['model']} (MAPE={row['mape']:.2f}%)")

SARIMA vs SARIMAX 성능 비교:
  model    family   mape      rmse       mae      aic  train_time_sec
SARIMAX BEVERAGES  49.70  58736.14  51467.66 32729.87            0.93
 SARIMA BEVERAGES  74.03  73701.07  65214.21 32762.09            1.30
SARIMAX  CLEANING 139.78  20910.02  17877.94 30166.13            1.36
 SARIMA  CLEANING 153.74  20400.08  17675.13 29833.95            1.23
SARIMAX     DAIRY  89.45  15252.72  13191.11 28656.41            0.59
 SARIMA     DAIRY  95.19  16835.86  14758.86 28400.62            0.56
SARIMAX GROCERY I  65.24  50684.57  37813.73 33910.77            1.50
 SARIMA GROCERY I 145.81 124337.61 116300.16 33945.38            0.90
SARIMAX   PRODUCE  79.78  29559.25  25267.42 31960.95            2.94
 SARIMA   PRODUCE  84.69  33466.22  29432.71 32107.32            1.10

상품군별 최적 모델 (MAPE 기준):
  BEVERAGES       → SARIMAX (MAPE=49.70%)
  CLEANING        → SARIMAX (MAPE=139.78%)
  DAIRY           → SARIMAX (MAPE=89.45%)
  GROCERY I       → SARIMAX (MAPE=65.24%)
  PRODUCE    

In [7]:
# 결과 저장
save_path = os.path.join(RES_DIR, "sarima_results.csv")
sarima_df.to_csv(save_path, index=False)
print(f"저장: {save_path}")

저장: E:\프로젝트\부족한 프로젝트 시작\수요예측 시스템\outputs\results\sarima_results.csv


---
## 4. 잔차 분석

상품군별 최적 모델의 잔차를 분석하여 모델 적합성을 검증한다.
잔차가 **백색잡음(white noise)**에 가까울수록 좋은 모델이다.

검증 4종:
1. 잔차 시계열 → 패턴 없이 랜덤
2. 잔차 히스토그램 → 정규분포 유사
3. 잔차 ACF → 자기상관 없음
4. Ljung-Box 검정 → p > 0.05

In [8]:
# 상품군별 최적 모델 선택 (MAPE 기준)
best_models = {}
for family in families:
    sub = sarima_df[sarima_df["family"] == family]
    best_model_name = sub.loc[sub["mape"].idxmin(), "model"]
    if best_model_name == "SARIMA":
        best_models[family] = sarima_models[family]
    else:
        best_models[family] = sarimax_models[family]

fig, axes = plt.subplots(len(families), 4, figsize=(20, 3.5 * len(families)))

ljungbox_results = []

for i, family in enumerate(families):
    model = best_models[family]
    resid = model.get_residuals().dropna()

    # 1. 잔차 시계열
    axes[i, 0].plot(resid.index, resid.values, linewidth=0.3, color="#2c3e50")
    axes[i, 0].axhline(y=0, color="red", linewidth=0.5)
    axes[i, 0].set_title(f"{family} - 잔차", fontsize=9)

    # 2. 히스토그램 + 정규분포
    axes[i, 1].hist(resid, bins=50, density=True, alpha=0.7, color="#3498db")
    x_norm = np.linspace(resid.min(), resid.max(), 100)
    axes[i, 1].plot(
        x_norm, stats.norm.pdf(x_norm, resid.mean(), resid.std()), "r-", linewidth=1.5
    )
    axes[i, 1].set_title(f"{family} - 히스토그램", fontsize=9)

    # 3. 잔차 ACF
    plot_acf(resid, ax=axes[i, 2], lags=30, alpha=0.05)
    axes[i, 2].set_title(f"{family} - 잔차 ACF", fontsize=9)

    # 4. Q-Q Plot
    stats.probplot(resid, dist="norm", plot=axes[i, 3])
    axes[i, 3].set_title(f"{family} - Q-Q Plot", fontsize=9)

    # Ljung-Box 검정
    lb = acorr_ljungbox(resid, lags=[7, 14, 21], return_df=True)
    for lag, row in lb.iterrows():
        ljungbox_results.append(
            {
                "family": family,
                "lag": lag,
                "lb_stat": row["lb_stat"],
                "lb_pvalue": row["lb_pvalue"],
                "white_noise": "Yes" if row["lb_pvalue"] > 0.05 else "No",
            }
        )

plt.suptitle("SARIMA 잔차 분석", fontsize=13, fontweight="bold")
plt.tight_layout()
save_path = os.path.join(FIG_DIR, "14_sarima_residuals.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"저장: {save_path}")

저장: E:\프로젝트\부족한 프로젝트 시작\수요예측 시스템\outputs\figures\14_sarima_residuals.png


In [9]:
# Ljung-Box 검정 결과
lb_df = pd.DataFrame(ljungbox_results)
print("Ljung-Box 검정 (p > 0.05 → 백색잡음):")
for family in families:
    sub = lb_df[lb_df["family"] == family]
    pvals = sub["lb_pvalue"].values
    wn = sub["white_noise"].values
    print(
        f"  {family:15s} → lag7 p={pvals[0]:.4f}({wn[0]}), "
        f"lag14 p={pvals[1]:.4f}({wn[1]}), lag21 p={pvals[2]:.4f}({wn[2]})"
    )

Ljung-Box 검정 (p > 0.05 → 백색잡음):
  BEVERAGES       → lag7 p=0.0000(No), lag14 p=0.0000(No), lag21 p=0.0000(No)
  CLEANING        → lag7 p=0.0522(Yes), lag14 p=0.0234(No), lag21 p=0.0061(No)
  DAIRY           → lag7 p=0.2991(Yes), lag14 p=0.0187(No), lag21 p=0.0128(No)
  GROCERY I       → lag7 p=0.1206(Yes), lag14 p=0.0015(No), lag21 p=0.0007(No)
  PRODUCE         → lag7 p=0.0236(No), lag14 p=0.0053(No), lag21 p=0.0199(No)


---
## 5. 예측 vs 실제 + 신뢰구간

In [10]:
fig, axes = plt.subplots(
    len(families), 1, figsize=(16, 3.5 * len(families)), sharex=True
)

for i, family in enumerate(families):
    model = best_models[family]
    val_sub = val[val["family"] == family].sort_values("date")
    steps = len(val_sub)

    # 95% CI 예측
    if model.exog_cols:
        pred_95 = model.predict(steps=steps, val_df=val, alpha=0.05)
        pred_80 = model.predict(steps=steps, val_df=val, alpha=0.20)
    else:
        pred_95 = model.predict(steps=steps, alpha=0.05)
        pred_80 = model.predict(steps=steps, alpha=0.20)

    y_true = val_sub["sales"].values[: len(pred_95)]
    dates = val_sub["date"].values[: len(pred_95)]

    # Train 마지막 30일
    train_tail = train[train["family"] == family].sort_values("date").tail(30)

    axes[i].plot(
        train_tail["date"],
        train_tail["sales"],
        color="gray",
        linewidth=1,
        label="Train",
    )
    axes[i].plot(dates, y_true, color="black", linewidth=1.2, label="실제")
    axes[i].plot(
        dates,
        pred_95["forecast"].values[: len(y_true)],
        color="#e74c3c",
        linewidth=1.2,
        label="SARIMA 예측",
    )
    axes[i].fill_between(
        dates,
        pred_95["lower_ci"].values[: len(y_true)],
        pred_95["upper_ci"].values[: len(y_true)],
        alpha=0.1,
        color="#e74c3c",
        label="95% CI",
    )
    axes[i].fill_between(
        dates,
        pred_80["lower_ci"].values[: len(y_true)],
        pred_80["upper_ci"].values[: len(y_true)],
        alpha=0.2,
        color="#e74c3c",
        label="80% CI",
    )
    axes[i].set_title(family, fontsize=11, fontweight="bold")
    axes[i].legend(loc="upper left", fontsize=8)

plt.suptitle("SARIMA 예측 vs 실제 (Validation 기간)", fontsize=13, fontweight="bold")
plt.tight_layout()
save_path = os.path.join(FIG_DIR, "13_sarima_forecast.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"저장: {save_path}")

저장: E:\프로젝트\부족한 프로젝트 시작\수요예측 시스템\outputs\figures\13_sarima_forecast.png


---
## 6. 베이스라인 대비 개선율

In [11]:
# 베이스라인 최적 vs SARIMA 최적 비교
best_baseline = baseline.loc[baseline.groupby("family")["mape"].idxmin()]
best_sarima = sarima_df.loc[sarima_df.groupby("family")["mape"].idxmin()]

comparison = []
for family in families:
    bl = best_baseline[best_baseline["family"] == family].iloc[0]
    sr = best_sarima[best_sarima["family"] == family].iloc[0]
    improvement = (bl["mape"] - sr["mape"]) / bl["mape"] * 100
    comparison.append(
        {
            "상품군": family,
            "베이스라인": f"{bl['model']} ({bl['mape']:.1f}%)",
            "SARIMA": f"{sr['model']} ({sr['mape']:.1f}%)",
            "MAPE 개선율": f"{improvement:+.1f}%",
            "RMSE 비교": f"{bl['rmse']:.0f} → {sr['rmse']:.0f}",
        }
    )

comp_df = pd.DataFrame(comparison)
print("베이스라인 vs SARIMA 비교:")
print(comp_df.to_string(index=False))

베이스라인 vs SARIMA 비교:
      상품군            베이스라인           SARIMA MAPE 개선율       RMSE 비교
BEVERAGES Naive_7d (44.0%)  SARIMAX (49.7%)   -13.0% 44454 → 58736
 CLEANING  MA_30d (120.0%) SARIMAX (139.8%)   -16.5% 15996 → 20910
    DAIRY   MA_30d (65.8%)  SARIMAX (89.5%)   -36.0% 11990 → 15253
GROCERY I    MA_7d (98.1%)  SARIMAX (65.2%)   +33.5% 59451 → 50685
  PRODUCE   MA_30d (62.1%)  SARIMAX (79.8%)   -28.4% 27180 → 29559


---
## 7. 요약 및 인사이트

In [12]:
print("=" * 60)
print("Day 3 SARIMA 분석 요약")
print("=" * 60)
print()
print("1. AIC 기반 최적 차수:")
for family in families:
    ar = auto_results[family]
    print(
        f"   {family:15s} → {ar['order']} x {ar['seasonal_order']}, AIC={ar['aic']:.0f}"
    )
print()
print("2. SARIMA 최적 MAPE:")
for _, row in best_sarima.iterrows():
    print(f"   {row['family']:15s} → {row['model']} MAPE={row['mape']:.2f}%")
print()
avg_sarima = best_sarima["mape"].mean()
avg_baseline = best_baseline["mape"].mean()
print(f"3. 전체 평균: 베이스라인 {avg_baseline:.1f}% → SARIMA {avg_sarima:.1f}%")
print(f"   개선율: {(avg_baseline - avg_sarima) / avg_baseline * 100:+.1f}%")
print()
print("4. 인사이트:")
print("   - 모든 상품군에서 SARIMAX(외생변수) > SARIMA(순수)")
print("   - GROCERY I에서 +33.5% 개선 (베이스라인 대비)")
print("   - 181일 장기 예측은 SARIMA의 한계, 단기 예측에 유리")
print("   - 다음 단계: Day 4 (Prophet), Day 5 (XGBoost)")

Day 3 SARIMA 분석 요약

1. AIC 기반 최적 차수:
   BEVERAGES       → (2, 1, 2) x (1, 1, 1, 7), AIC=32762
   CLEANING        → (2, 1, 1) x (1, 1, 1, 7), AIC=29834
   DAIRY           → (1, 1, 1) x (0, 1, 1, 7), AIC=28401
   GROCERY I       → (2, 1, 2) x (1, 1, 1, 7), AIC=33945
   PRODUCE         → (2, 1, 2) x (1, 1, 1, 7), AIC=32107

2. SARIMA 최적 MAPE:
   BEVERAGES       → SARIMAX MAPE=49.70%
   CLEANING        → SARIMAX MAPE=139.78%
   DAIRY           → SARIMAX MAPE=89.45%
   GROCERY I       → SARIMAX MAPE=65.24%
   PRODUCE         → SARIMAX MAPE=79.78%

3. 전체 평균: 베이스라인 78.0% → SARIMA 84.8%
   개선율: -8.7%

4. 인사이트:
   - 모든 상품군에서 SARIMAX(외생변수) > SARIMA(순수)
   - GROCERY I에서 +33.5% 개선 (베이스라인 대비)
   - 181일 장기 예측은 SARIMA의 한계, 단기 예측에 유리
   - 다음 단계: Day 4 (Prophet), Day 5 (XGBoost)
